<a href="https://colab.research.google.com/github/Midushii/AI_vs_Human_ChildMedia/blob/main/Ai_Features.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q librosa soundfile pandas numpy tqdm opencv-python-headless

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

BASE_DRIVE_PATH = "/content/drive/MyDrive/ai generated kids content"
AI_FOLDER = os.path.join(BASE_DRIVE_PATH, "short")

OUTPUT_DIR = os.path.join(BASE_DRIVE_PATH, "features_output")
os.makedirs(OUTPUT_DIR, exist_ok=True)

AI_CSV = os.path.join(OUTPUT_DIR, "ai_features_v3.csv")

VIDEO_EXTS = {".mp4", ".mkv", ".webm", ".mov"}
FRAME_SAMPLE_FPS = 1
SCENE_THRESHOLD = 0.3

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import cv2
import numpy as np
import pandas as pd
import librosa
import soundfile as sf
import subprocess
import tempfile
import shutil
import re
from pathlib import Path
from tqdm import tqdm

In [ ]:
def get_video_id(filename):
    return Path(filename).stem[:11]

In [ ]:
def extract_pacing_features(video_path):
    """
    Uses ffmpeg's built-in scene-change filter instead of PySceneDetect/cv2 --
    this is the same ffmpeg binary that has opened 100% of your files
    successfully for audio extraction, so it sidesteps the codec-support gap
    that was causing cv2 to silently fail on most videos.
    """
    try:
        # Get duration first via ffprobe (fast, metadata-only)
        probe_cmd = ["ffprobe", "-v", "error", "-show_entries", "format=duration",
                     "-of", "default=noprint_wrappers=1:nokey=1", str(video_path)]
        probe_result = subprocess.run(probe_cmd, capture_output=True, text=True)
        duration_sec = float(probe_result.stdout.strip())

        # Run ffmpeg scene-detection filter, capture timestamps from stderr
        cmd = [
            "ffmpeg", "-i", str(video_path),
            "-filter:v", f"select='gt(scene,{SCENE_THRESHOLD})',showinfo",
            "-f", "null", "-"
        ]
        result = subprocess.run(cmd, capture_output=True, text=True)
        timestamps = re.findall(r"pts_time:([\d.]+)", result.stderr)
        num_cuts = len(timestamps)

        if duration_sec <= 0:
            return {"shot_count_per_sec": None, "avg_shot_duration_sec": None,
                    "num_cuts": None, "duration_sec": None}

        shot_count_per_sec = num_cuts / duration_sec
        avg_shot_duration_sec = duration_sec / num_cuts if num_cuts > 0 else duration_sec

        return {
            "shot_count_per_sec": round(shot_count_per_sec, 4),
            "avg_shot_duration_sec": round(avg_shot_duration_sec, 3),
            "num_cuts": num_cuts,
            "duration_sec": round(duration_sec, 3),
        }
    except Exception as e:
        print(f"    [pacing failed for {video_path.name}: {e}]")
        return {"shot_count_per_sec": None, "avg_shot_duration_sec": None,
                "num_cuts": None, "duration_sec": None}

In [ ]:
def extract_frames_via_ffmpeg(video_path, out_dir, fps=FRAME_SAMPLE_FPS):
    cmd = [
        "ffmpeg", "-i", str(video_path), "-vf", f"fps={fps}",
        "-q:v", "2", "-loglevel", "error",
        os.path.join(out_dir, "frame_%05d.jpg")
    ]
    result = subprocess.run(cmd, capture_output=True)
    if result.returncode != 0:
        raise RuntimeError(f"ffmpeg frame extraction failed: {result.stderr.decode(errors='ignore')[:200]}")


def shannon_entropy(gray_img):
    hist = cv2.calcHist([gray_img], [0], None, [256], [0, 256]).flatten()
    hist = hist / (hist.sum() + 1e-9)
    hist = hist[hist > 0]
    return float(-np.sum(hist * np.log2(hist)))


def extract_visual_features(video_path):
    tmp_dir = tempfile.mkdtemp()
    try:
        extract_frames_via_ffmpeg(video_path, tmp_dir)
        frame_files = sorted(Path(tmp_dir).glob("frame_*.jpg"))

        if not frame_files:
            print(f"    [visual: no frames extracted for {video_path.name}]")
            return {"brightness_mean": None, "saturation_mean": None,
                    "color_warmness_pct": None, "visual_complexity": None}

        brightness_vals, sat_vals, warm_vals, complexity_vals = [], [], [], []

        for fpath in frame_files:
            frame = cv2.imread(str(fpath))   # static image read -- no video codec involved
            if frame is None:
                continue

            hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
            h, s, v = hsv[:, :, 0], hsv[:, :, 1], hsv[:, :, 2]

            brightness_vals.append(np.mean(v))
            sat_vals.append(np.mean(s))

            # Paper's definition: warm hue strictly 0-30 (red to yellow), out of 0-255 all pixels
            warm_mask = (h <= 30)
            warm_pct = float(np.sum(warm_mask)) / warm_mask.size
            warm_vals.append(warm_pct)

            gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            complexity_vals.append(shannon_entropy(gray))

        if not brightness_vals:
            return {"brightness_mean": None, "saturation_mean": None,
                    "color_warmness_pct": None, "visual_complexity": None}

        return {
            "brightness_mean": round(float(np.mean(brightness_vals)), 3),
            "saturation_mean": round(float(np.mean(sat_vals)), 3),
            "color_warmness_pct": round(float(np.mean(warm_vals)), 4),
            "visual_complexity": round(float(np.mean(complexity_vals)), 3),
        }
    except Exception as e:
        print(f"    [visual failed for {video_path.name}: {e}]")
        return {"brightness_mean": None, "saturation_mean": None,
                "color_warmness_pct": None, "visual_complexity": None}
    finally:
        shutil.rmtree(tmp_dir, ignore_errors=True)

In [ ]:
def extract_audio_to_wav(video_path, out_sr=48000):
    tmp_wav = tempfile.NamedTemporaryFile(suffix=".wav", delete=False).name
    cmd = ["ffmpeg", "-y", "-i", str(video_path), "-ac", "1", "-ar", str(out_sr),
           "-vn", "-loglevel", "error", tmp_wav]
    result = subprocess.run(cmd, capture_output=True)
    if result.returncode != 0 or not os.path.exists(tmp_wav):
        raise RuntimeError(f"ffmpeg audio extraction failed: {result.stderr.decode(errors='ignore')[:200]}")
    return tmp_wav


def extract_audio_features(video_path):
    result = {"loudness": None, "tempo_bpm": None, "sound_brightness": None}
    tmp_wav = None
    try:
        tmp_wav = extract_audio_to_wav(video_path)
        y, sr = sf.read(tmp_wav)
        if y.ndim > 1:
            y = y.mean(axis=1)
        if y.size == 0:
            return result

        # Loudness -- paper's definition: mean RMS energy, naturally ~0-1 for
        # normalized waveform amplitude (not LUFS, to match the paper exactly)
        rms = librosa.feature.rms(y=y.astype(np.float32))[0]
        result["loudness"] = round(float(np.mean(rms)), 4)

        try:
            tempo, _ = librosa.beat.beat_track(y=y.astype(np.float32), sr=sr)
            result["tempo_bpm"] = round(float(tempo), 2)
        except Exception as e:
            print(f"    [tempo failed for {video_path.name}: {e}]")

        try:
            centroid = librosa.feature.spectral_centroid(y=y.astype(np.float32), sr=sr)
            result["sound_brightness"] = round(float(np.mean(centroid)), 2)
        except Exception as e:
            print(f"    [spectral centroid failed for {video_path.name}: {e}]")

    except Exception as e:
        print(f"    [audio extraction failed for {video_path.name}: {e}]")
    finally:
        if tmp_wav and os.path.exists(tmp_wav):
            os.remove(tmp_wav)

    return result

In [ ]:
def _save_rows(rows, output_csv):
    if not rows:
        return
    new_df = pd.DataFrame(rows)
    if os.path.exists(output_csv):
        old_df = pd.read_csv(output_csv)
        combined = pd.concat([old_df, new_df], ignore_index=True).drop_duplicates(subset="video_id", keep="last")
    else:
        combined = new_df
    combined.to_csv(output_csv, index=False)


def process_folder(folder_path, output_csv, label):
    folder_path = Path(folder_path)
    video_files = [p for p in folder_path.iterdir() if p.suffix.lower() in VIDEO_EXTS]
    print(f"Found {len(video_files)} videos in {folder_path}\n")

    already_done = set()
    if os.path.exists(output_csv):
        already_done = set(pd.read_csv(output_csv)["video_id"].astype(str))
        print(f"  {len(already_done)} already processed -- skipping those\n")

    for i, video_path in enumerate(video_files, 1):
        vid = get_video_id(video_path.name)
        if vid in already_done:
            continue

        row = {"video_id": vid, "filename": video_path.name, "label": label}
        row.update(extract_pacing_features(video_path))
        row.update(extract_visual_features(video_path))
        row.update(extract_audio_features(video_path))

        # Save to Drive IMMEDIATELY -- one video at a time, not batched
        _save_rows([row], output_csv)

        # Print this video's result right now so you can eyeball it live
        status = "OK" if row.get("brightness_mean") is not None and row.get("num_cuts", 0) > 0 else "CHECK THIS"
        print(f"[{i}/{len(video_files)}] {video_path.name[:60]}")
        print(f"    cuts={row.get('num_cuts')}  brightness={row.get('brightness_mean')}  "
              f"saturation={row.get('saturation_mean')}  warmth={row.get('color_warmness_pct')}  "
              f"complexity={row.get('visual_complexity')}  loudness={row.get('loudness')}  "
              f"tempo={row.get('tempo_bpm')}  sound_brightness={row.get('sound_brightness')}  "
              f"--> {status}")
        print()

In [ ]:
process_folder(AI_FOLDER, AI_CSV, label="ai")

Found 290 videos in /content/drive/MyDrive/ai generated kids content/short



/tmp/ipykernel_4027/515548537.py:29: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  result["tempo_bpm"] = round(float(tempo), 2)


[1/290] 6lRy1ZZQtes_A Fox Saved a Lost Duckling ❤️ ｜ Emotional Anima
    cuts=83  brightness=148.352  saturation=148.439  warmth=0.419  complexity=7.698  loudness=0.4449  tempo=144.23  sound_brightness=2057.82  --> OK



/tmp/ipykernel_4027/515548537.py:29: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  result["tempo_bpm"] = round(float(tempo), 2)


[2/290] ba62uuv-5Dc_Rollin' France - what if animals were round？.mp4
    cuts=5  brightness=111.368  saturation=94.276  warmth=0.3995  complexity=7.061  loudness=0.0511  tempo=112.5  sound_brightness=4400.59  --> OK



...

[288/290] owL0niaTj44_Beautiful mountain village background｜ #backgrou
    cuts=0  brightness=150.364  saturation=103.955  warmth=0.1847  complexity=7.72  loudness=0.0945  tempo=165.44  sound_brightness=1908.01  --> CHECK THIS



/tmp/ipykernel_4027/515548537.py:29: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  result["tempo_bpm"] = round(float(tempo), 2)


[289/290] hxCe43B3l8k_AI animation cat video #cats.mp4
    cuts=6  brightness=116.211  saturation=127.243  warmth=0.8246  complexity=7.592  loudness=0.0441  tempo=144.23  sound_brightness=1990.85  --> OK



/tmp/ipykernel_4027/515548537.py:29: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  result["tempo_bpm"] = round(float(tempo), 2)


[290/290] JWvYVVMWdcM_Animal Story AI ｜ Emotional & Moral Animal Stori
    cuts=7  brightness=99.132  saturation=81.33  warmth=0.7047  complexity=6.071  loudness=0.3459  tempo=108.17  sound_brightness=2215.01  --> OK

